In [1]:
import pandas as pd
import numpy as np
import re


from unicodedata import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering, DBSCAN

from sklearn.cluster import AffinityPropagation
from sklearn.metrics.pairwise import cosine_similarity

from difflib import SequenceMatcher


In [2]:
path = "data/train/cleaned_train.csv"

df = pd.read_csv(path)

# Clusterización de versiones


## Experimentos para un modelo especificado

In [3]:
brand = "Ford"
model = "Ecosport"

version_df = df[(df["Marca"] == brand) & (df["Modelo"] == model)].reset_index(drop=True)

### Limpieza y preprocesamiento de versiones

Este código se encarga de limpiar y normalizar los nombres de las versiones de vehículos para facilitar su posterior análisis y agrupación:

**Función** `clean_version()`
- **Normalización de caracteres**: Convierte texto a minúsculas y elimina acentos/caracteres especiales usando Unicode NFKD
- **Filtrado de caracteres**: Solo mantiene letras, números y espacios, eliminando símbolos y puntuación
- **Limpieza de espacios**: Normaliza espacios múltiples a espacios únicos y elimina espacios al inicio/final
- **Manejo de valores nulos**: Convierte valores `NaN` a strings vacíos


In [4]:
def clean_version(s: str) -> str:
    s = "" if pd.isna(s) else s
    s = normalize("NFKD", s.lower()).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

version_df["version_clean"] = version_df["Versión"].apply(clean_version)
version_df = version_df[version_df["version_clean"] != ""].reset_index(drop=True)

print("Número de versiones únicas:", version_df["version_clean"].nunique())

print("\nPrimeras 5 versiones únicas:")
print(version_df["version_clean"].value_counts()[:5])

Número de versiones únicas: 49

Primeras 5 versiones únicas:
version_clean
se           375
titanium     259
freestyle    144
s             92
my10 xls      46
Name: count, dtype: int64


### HAC y TF-IDF

In [5]:
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"])

non_zero_mask = np.array(X.sum(axis=1)).flatten() > 0
X_filtered = X[non_zero_mask].toarray()
taos_df_filtered = version_df[non_zero_mask].copy()

agg = AgglomerativeClustering(
    metric="cosine",
    linkage="average",
    distance_threshold=0.8,
    n_clusters=None,
)

# Fit on filtered data
clusters_filtered = agg.fit_predict(X_filtered)

# Assign clusters back to original dataframe
version_df["cluster"] = -1  # Default for zero vectors
version_df.loc[non_zero_mask, "cluster"] = clusters_filtered

In [6]:
canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary

,cluster,version_canon,count
0,-1,se,467
1,0,titanium,323
3,2,freestyle,178
2,1,my10 xls,140
13,12,xls,42
5,4,xls mp3,28
7,6,se tdci,20
8,7,gdi storm,12
10,9,d se,9
4,3,se l18,6


### Affinity propagation

In [7]:
S = cosine_similarity(X)

pref = np.median(S) - 0.05
ap = AffinityPropagation(affinity="precomputed",
                         damping=0.95,
                         preference=pref,
                         max_iter=500,
                         convergence_iter=16)

version_df["cluster"] = ap.fit_predict(S)


canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
summary


,cluster,version_canon,count
3,3,se,341
1,1,se,311
0,0,se,300
2,2,se,285


### DBSCAN

- Leer `CountVectorizer` en [Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) (podemos jugar con los argumentos)

In [8]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"]).toarray()

print(X.shape)

(1237, 377)


In [9]:
n_versions = version_df["version_clean"].nunique()

db = DBSCAN(eps=0.3, min_samples=max(2, n_versions // 4), metric="cosine")
version_df["cluster"] = db.fit_predict(X)

# canónico solo para clusters válidos (≠ –1)
canon = (
    version_df[version_df["cluster"] != -1]
      .groupby("cluster")["Versión"]
      .agg(lambda s: s.value_counts().idxmax())
)
version_df["version_canon"] = version_df["cluster"].map(canon)

# todo el ruido recibe una misma etiqueta
version_df.loc[version_df["cluster"] == -1, "version_canon"] = "Unassigned"

summary = (
    version_df.groupby(["cluster", "version_canon"])
              .size()
              .reset_index(name="count")
              .sort_values("count", ascending=False)
)
print(summary)
print("\nClusters densos:", canon.size,
      "  Etiquetas finales:", version_df["version_canon"].nunique())


   cluster     version_canon  count
0       -1        Unassigned    521
1        0          titanium    321
3        2         freestyle    178
5        4  my10 xl plus mp3     60
7        6          my10 xls     59
6        5               xls     42
4        3          xlt plus     21
8        7           se tdci     18
2        1           xls mp3     17

Clusters densos: 8   Etiquetas finales: 9


### Usar los datos de `mercadolibre_versions.csv`

In [10]:
ml_versions = pd.read_csv('data/mercadolibre_versions.csv')
ford_ml = ml_versions[(ml_versions['Brand'] == brand) &
                      (ml_versions['Model'] == model ) &
                      (ml_versions['Version'] != 'N/A')]

real_versions = ford_ml['Version'].tolist()

clustered_counts = version_df['Versión'].value_counts()
print(clustered_counts.head())

threshold = 0.3
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3), lowercase=True)

final_mapping = {}
mapping_results = []

for clustered in clustered_counts.index:

    best_match = None
    best_score = 0.0
    clean_clustered = clean_version(clustered)


    for rv in real_versions:
        clean_real = clean_version(rv)
        if clean_clustered and clean_real:
            X = vectorizer.fit_transform([clean_clustered, clean_real])
            score = cosine_similarity(X[0:1], X[1:2])[0][0]
        else:
            score = 0.0
        if score > best_score:
            best_score = score
            best_match = rv
    if best_score < threshold:
        best_match = 'Se'
        
    final_mapping[clustered] = best_match
    mapping_results.append({
        'clustered_version': clustered,
        'count': clustered_counts[clustered],
        'mapped_to': best_match,
        'similarity_score': best_score
    })

mapping_df = pd.DataFrame(mapping_results)
print(mapping_df.to_string(index=False))

Versión
se           375
titanium     259
freestyle    144
s             92
my10 xls      46
Name: count, dtype: int64
       clustered_version  count mapped_to  similarity_score
                      se    375        Se          1.000000
                titanium    259  Titanium          1.000000
               freestyle    144 Freestyle          1.000000
                       s     92         S          1.000000
                my10 xls     46  My10 Xls          1.000000
                     xls     42       Xls          1.000000
        my10 xl plus mp3     33  My10 Xls          0.664784
            gdi titanium     28  Titanium          0.843338
           freestyle 143     22 Freestyle          0.845154
titanium powershift dps6     18  Titanium          0.621825
             xl plus mp3     18   Xls Mp3          0.707107
                 se tdci     18        Se          0.408248
                 xls mp3     17   Xls Mp3          1.000000
           my10 xls plus     12  My10 Xls

In [11]:
print("Before mapping:")
print("Original version counts:", version_df["Versión"].value_counts())
print(f"Total unique versions: {version_df['Versión'].nunique()}")

version_df["Versión"] = version_df["Versión"].map(final_mapping)

print("\nAfter mapping:")
print("Mapped version counts:", version_df["Versión"].value_counts())
print(f"Total unique versions: {version_df['Versión'].nunique()}")

# Verify that all versions are now real MercadoLibre versions
mapped_versions = set(version_df["Versión"].unique())
real_versions_set = set(real_versions)

print(f"\nValidation:")
print(f"All mapped versions are real versions: {mapped_versions.issubset(real_versions_set)}")
print(f"Mapped versions: {sorted(mapped_versions)}")
print(f"Real versions not used: {sorted(real_versions_set - mapped_versions)}")

# Show final summary
final_summary = version_df["Versión"].value_counts().reset_index()
final_summary.columns = ["Version", "Count"]
final_summary["Percentage"] = (final_summary["Count"] / final_summary["Count"].sum() * 100).round(2)

print(f"\nFinal Version Distribution:")
print("="*50)
print(final_summary.to_string(index=False))

Before mapping:
Original version counts: Versión
se                          375
titanium                    259
freestyle                   144
s                            92
my10 xls                     46
xls                          42
my10 xl plus mp3             33
gdi titanium                 28
freestyle 143                22
titanium powershift dps6     18
xl plus mp3                  18
se tdci                      18
xls mp3                      17
my10 xls plus                12
xlt plus                     11
my10 xlt plus                10
d se                          9
xl plus                       8
gdi storm                     7
titanium powershift           6
gdi freestyle                 6
tdci xls mp3                  5
xlt                           5
se l18                        4
tdci xls                      4
gdi se                        4
titanium l18                  3
freestyle l18                 3
gdi                           3
xls seguridad          

## Asignar cada muestra al cluster que pertenece

In [12]:
version_df['Versión'] = version_df['Versión'].map(final_mapping)

summary = version_df['Versión'].value_counts().reset_index()
summary.columns = ['Version', 'Count']
summary['Percentage'] = (summary['Count'] / summary['Count'].sum() * 100).round(2)

print(summary.to_string(index=False))


Empty DataFrame
Columns: [Version, Count, Percentage]
Index: []


# Compartir features entre cada versión

Ejemplo: Si un auto en particular de una cierta versión tiene turbo, y los demás autos de esa versión no lo especifican, entonces se le asigna turbo a los demás autos de esa versión.

In [13]:
def fill_by_mode(df, col, keys=("Marca", "Modelo", "Versión")):
    mode_vals = (
        df.groupby(list(keys))[col]
          .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    )
    df[col] = df[col].fillna(mode_vals)

cols_to_fill = ["cv", "Motor", "Tracción", "Turbo"]

for c in cols_to_fill:
    if c in version_df.columns:
        fill_by_mode(version_df, c)

print("Faltantes después de imputar:")
print(version_df[cols_to_fill].isna().sum())

Faltantes después de imputar:
cv          319
Motor         1
Tracción     99
Turbo         0
dtype: int64


# Color

In [14]:
# Análisis inicial de colores
color_counts = df['Color'].value_counts()
print("Cantidad de colores únicos:", len(color_counts), "\n")

with pd.option_context('display.max_rows', None):
    print(color_counts)

Cantidad de colores únicos: 64 

Color
Gris                             4395
Blanco                           3817
Negro                            2264
Plateado                         1132
Rojo                              904
Azul                              887
Marrón                            213
Dorado                            150
Verde                             107
Beige                             100
Celeste                            74
Naranja                            58
Gris oscuro                        35
Bordó                              28
Violeta                            24
Amarillo                           21
Rosa                                7
Plata                               5
Otro                                5
Acero                               3
Blanca                              3
Negra                               3
Gris plata                          3
PLATA                               2
MINERALWEISS METALLIC               2
Café       

In [15]:
# Función para limpiar y normalizar colores
def clean_color(color):
    if pd.isna(color):
        return ""
    
    # Convertir a string y normalizar
    color_str = str(color).strip().lower()
    
    # Normalizar caracteres especiales
    color_str = normalize("NFKD", color_str).encode("ascii", "ignore").decode("ascii")
    
    # Eliminar caracteres especiales excepto letras, números y espacios
    color_str = re.sub(r"[^a-z0-9 ]+", " ", color_str)
    
    # Normalizar espacios
    color_str = re.sub(r"\s+", " ", color_str).strip()
    
    return color_str

# Aplicar limpieza a la columna Color
df['Color_clean'] = df['Color'].apply(clean_color)

# Verificar colores únicos después de la limpieza
color_counts_clean = df['Color_clean'].value_counts()
print("Cantidad de colores únicos después de limpieza:", len(color_counts_clean), "\n")
print("Top 15 colores más frecuentes:")
print(color_counts_clean.head(15))

Cantidad de colores únicos después de limpieza: 60 

Top 15 colores más frecuentes:
Color_clean
gris           4395
blanco         3817
negro          2264
plateado       1132
rojo            904
azul            887
                315
marron          213
dorado          150
verde           107
beige           100
celeste          74
naranja          58
gris oscuro      35
bordo            28
Name: count, dtype: int64


In [16]:
# Definir el número N de colores más frecuentes a mantener
N = 8

# Obtener los N colores más frecuentes
top_colors = color_counts_clean.head(N).index.tolist()
print(f"Los {N} colores más frecuentes que se mantendrán:")
for i, color in enumerate(top_colors, 1):
    print(f"{i}. {color} ({color_counts_clean[color]} casos)")

Los 8 colores más frecuentes que se mantendrán:
1. gris (4395 casos)
2. blanco (3817 casos)
3. negro (2264 casos)
4. plateado (1132 casos)
5. rojo (904 casos)
6. azul (887 casos)
7.  (315 casos)
8. marron (213 casos)


### Entrenar modelo de Word2Vec

In [17]:
# Preparar lista de todos los colores únicos
all_colors = [color for color in df['Color_clean'].unique() if color and color not in ['', 'nan']]

print(f"Total de colores únicos para vectorizar: {len(all_colors)}")
print("Ejemplos de colores:")
for i in range(min(10, len(all_colors))):
    print(f"  {i+1}. {all_colors[i]}")

# Crear y entrenar CountVectorizer
# Usamos n-gramas de caracteres para capturar similitudes entre colores
vectorizer = CountVectorizer(
    analyzer='char',
    ngram_range=(2, 4),  # N-gramas de 2 a 4 caracteres
    lowercase=True
)

# Ajustar el vectorizador con todos los colores
color_vectors = vectorizer.fit_transform(all_colors)

print(f"\nDimensiones de la matriz de vectores: {color_vectors.shape}")
print(f"Características extraídas: {len(vectorizer.get_feature_names_out())}")

Total de colores únicos para vectorizar: 59
Ejemplos de colores:
  1. rojo
  2. negro
  3. plateado
  4. blanco
  5. gris
  6. azul
  7. verde
  8. blanca
  9. dorado
  10. beige

Dimensiones de la matriz de vectores: (59, 792)
Características extraídas: 792


In [18]:
# Función para obtener el índice de un color en la lista
def get_color_index(color, color_list):
    """Obtiene el índice de un color en la lista"""
    try:
        return color_list.index(color)
    except ValueError:
        return -1

# Función para encontrar el color más similar
def find_most_similar_color(target_color, top_colors, all_colors, color_vectors):
    """Encuentra el color más similar entre los colores top usando CountVectorizer"""
    if target_color in top_colors:
        return target_color  # Si ya está en los top, no cambiar
    
    # Obtener índice del color objetivo
    target_idx = get_color_index(target_color, all_colors)
    
    if target_idx == -1:
        # Si no se encuentra el color, asignar al más frecuente
        return top_colors[0]
    
    # Obtener vector del color objetivo
    target_vector = color_vectors[target_idx]
    
    # Calcular similitudes con todos los colores top
    similarities = {}
    for top_color in top_colors:
        top_idx = get_color_index(top_color, all_colors)
        if top_idx != -1:
            top_vector = color_vectors[top_idx]
            similarity = cosine_similarity(target_vector, top_vector)[0][0]
            similarities[top_color] = similarity
    
    # Retornar el color con mayor similitud
    if similarities:
        return max(similarities, key=similarities.get)
    else:
        return top_colors[0]  # Fallback al más frecuente

### Aplicar clustering de colores

In [19]:
# Aplicar la clusterización de colores
print("Aplicando clusterización de colores...")

# Crear diccionario de mapeo para colores no top
color_mapping = {}
colors_to_process = [color for color in df['Color_clean'].unique() 
                    if color and color not in ['', 'nan'] and color not in top_colors]

print(f"Procesando {len(colors_to_process)} colores únicos que no están en los top {N}...")

for color in colors_to_process:
    most_similar = find_most_similar_color(color, top_colors, all_colors, color_vectors)
    color_mapping[color] = most_similar
    print(f"  '{color}' -> '{most_similar}'")

print("\nMapeo completado!")

Aplicando clusterización de colores...
Procesando 52 colores únicos que no están en los top 8...
  'verde' -> 'gris'
  'blanca' -> 'blanco'
  'dorado' -> 'plateado'
  'beige' -> 'gris'
  'celeste' -> 'plateado'
  'gris oscuro' -> 'gris'
  'bordo' -> 'plateado'
  'marron oscuro' -> 'marron'
  'rosa' -> 'rojo'
  'gris titane' -> 'gris'
  'naranja' -> 'blanco'
  'gris plata' -> 'gris'
  'violeta' -> 'gris'
  'grafito' -> 'gris'
  'amarillo' -> 'marron'
  'verde oscuro' -> 'rojo'
  'blanco glaciar' -> 'blanco'
  'negra' -> 'negro'
  'carbon black' -> 'blanco'
  'skyscraper grau metallic' -> 'gris'
  'plata' -> 'plateado'
  'cafe' -> 'gris'
  'plata bari' -> 'plateado'
  'mineralweiss metallic' -> 'gris'
  'acero' -> 'rojo'
  'gris silverstone' -> 'gris'
  'prata bari tet vulc' -> 'plateado'
  'granite crysta bc' -> 'gris'
  'otro' -> 'rojo'
  'cendre' -> 'gris'
  'beige techo negro' -> 'negro'
  'rojo sunset metalizado' -> 'rojo'
  'marron claro' -> 'marron'
  'summit white' -> 'plateado'


In [20]:
# Mostrar similitudes detalladas para algunos ejemplos
print("Similitudes detalladas para algunos colores de ejemplo:")

example_colors = colors_to_process[:5]  # Tomar los primeros 5 como ejemplo

for target_color in example_colors:
    target_idx = get_color_index(target_color, all_colors)
    if target_idx != -1:
        target_vector = color_vectors[target_idx]
        
        print(f"\nColor: '{target_color}'")
        print("Similitudes con colores top:")
        
        similarities = []
        for top_color in top_colors:
            top_idx = get_color_index(top_color, all_colors)
            if top_idx != -1:
                top_vector = color_vectors[top_idx]
                similarity = cosine_similarity(target_vector, top_vector)[0][0]
                similarities.append((top_color, similarity))
        
        # Ordenar por similitud descendente
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        for top_color, sim in similarities:
            print(f"  {top_color}: {sim:.4f}")
        
        print(f"  -> Asignado a: '{similarities[0][0]}'")

Similitudes detalladas para algunos colores de ejemplo:

Color: 'verde'
Similitudes con colores top:
  gris: 0.0000
  blanco: 0.0000
  negro: 0.0000
  plateado: 0.0000
  rojo: 0.0000
  azul: 0.0000
  marron: 0.0000
  -> Asignado a: 'gris'

Color: 'blanca'
Similitudes con colores top:
  blanco: 0.7500
  plateado: 0.0680
  gris: 0.0000
  negro: 0.0000
  rojo: 0.0000
  azul: 0.0000
  marron: 0.0000
  -> Asignado a: 'blanco'

Color: 'dorado'
Similitudes con colores top:
  plateado: 0.2520
  gris: 0.0000
  blanco: 0.0000
  negro: 0.0000
  rojo: 0.0000
  azul: 0.0000
  marron: 0.0000
  -> Asignado a: 'plateado'

Color: 'beige'
Similitudes con colores top:
  gris: 0.0000
  blanco: 0.0000
  negro: 0.0000
  plateado: 0.0000
  rojo: 0.0000
  azul: 0.0000
  marron: 0.0000
  -> Asignado a: 'gris'

Color: 'celeste'
Similitudes con colores top:
  plateado: 0.0609
  gris: 0.0000
  blanco: 0.0000
  negro: 0.0000
  rojo: 0.0000
  azul: 0.0000
  marron: 0.0000
  -> Asignado a: 'plateado'


In [21]:
# Aplicar el mapeo al dataset
df['Color_clustered'] = df['Color_clean'].apply(
    lambda x: color_mapping.get(x, x) if x in color_mapping else x
)

# Verificar resultados
print("Resultados de la clusterización:")
print(f"Colores únicos originales: {len(color_counts)}")
print(f"Colores únicos después de clusterización: {df['Color_clustered'].nunique()}")

print("\nDistribución de colores después de clusterización:")
clustered_counts = df['Color_clustered'].value_counts()
print(clustered_counts)

Resultados de la clusterización:
Colores únicos originales: 64
Colores únicos después de clusterización: 8

Distribución de colores después de clusterización:
Color_clustered
gris        4681
blanco      3891
negro       2271
plateado    1396
rojo         922
azul         889
             315
marron       238
Name: count, dtype: int64


In [22]:
# Análisis detallado de los cambios
print("Análisis detallado de los cambios realizados:")

# Crear DataFrame de comparación
comparison_df = pd.DataFrame({
    'Color_original': df['Color_clean'],
    'Color_clustered': df['Color_clustered']
})

# Identificar filas donde cambió el color
changed_mask = comparison_df['Color_original'] != comparison_df['Color_clustered']
changed_rows = comparison_df[changed_mask]

print(f"Total de registros donde cambió el color: {len(changed_rows)}")
print(f"Porcentaje de cambios: {(len(changed_rows) / len(df)) * 100:.2f}%")

if len(changed_rows) > 0:
    print("\nEjemplos de cambios realizados:")
    changes_summary = changed_rows.groupby(['Color_original', 'Color_clustered']).size().reset_index(name='count')
    changes_summary = changes_summary.sort_values('count', ascending=False)
    print(changes_summary.head(10))

Análisis detallado de los cambios realizados:
Total de registros donde cambió el color: 676
Porcentaje de cambios: 4.63%

Ejemplos de cambios realizados:
   Color_original Color_clustered  count
18         dorado        plateado    150
49          verde            gris    107
4           beige            gris    100
15        celeste        plateado     74
37        naranja          blanco     58
26    gris oscuro            gris     35
12          bordo        plateado     28
51        violeta            gris     24
1        amarillo          marron     21
45           rosa            rojo      7
